# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. Выведит: 20, 10. В индексе b у нас лежит 20, а на нулевой позиции лежит 10
2. Выведет: Nan, Nan, 3, так как в а нету Астаны, а в b нету Алматы
3. Выведет: 2 2, так как после маски у нас остаются 1 и 2 ярлык
4. Выведет: Series и DataFrame, так в одном слое выдаст series, а в двух квадратных скобках даст DataFrame
5. Выведет: [1, 2] 5. f.index.tolist() - возьмёт все индексы и превратит их в список, а f.iloc[0, 1] возьмёт первую строку и второй элемент этой строки

In [9]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [11]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [26]:
# 1
print(f"Строк и столбцов: {df.shape}")
print(df.info())
print("-------")
print(f"Уникальных значение в city: {df["city"].nunique()}, в category: {df["category"].nunique()}, в status: {df["status"].nunique()}, в customer_id: {df["customer_id"].nunique()},")
df["city"].unique()

Строк и столбцов: (5463, 8)
<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 671.2 KB
None
-------
Уникальных значение в city: 14, в category: 5, в status: 2, в customer_id: 2003,


<ArrowStringArray>
[   'Алматы',    'Астана', 'Караганда',   'Шымкент',    'алматы',    'Атырау',
   'шымкент',  'Shymkent',    'астана',    'Актобе',   'Astana ',    'АЛМАТЫ',
   ' Алматы',    'Almaty']
Length: 14, dtype: str

Исходя из данного анализа, первое что можно увидеть, что в категории городов у нас 14 уникальных элементов, но если пригледется, то можно понять что города написаны в разных регистрах и в разных языках, что в свою очередь может привести к тому, чтобы не сможем посчитать правильно доход каждого города и его долю. 
Вторая проблема заключается в том, что у нас есть пропуски в данных в категории доставки, что в свою очередь может не дать точного анализа.

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [52]:
# 2
clear_df = df[paid & not_test]
revenue_info = clear_df["amount"].describe()
print(f"Типичный чек: {revenue_info["50%"]}")
revenue_info

Типичный чек: 10300.0


count      4901.000000
mean      14672.391349
std       13249.340787
min         720.000000
25%        5880.000000
50%       10300.000000
75%       18690.000000
max      139040.000000
Name: amount, dtype: float64

Честнее всего медиана, так как медиана берет значение по середине отсортированого списка значений. А в средней могут быть выбросы, что в свою очередь может дать не точный средний чек. 

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [24]:
# 3
AL = []
for i in df["city"].unique():
    if i.replace(" ", "").lower() == "алматы":
        AL.append(i)
    elif i.replace(" ", "").lower() == "almaty":
        AL.append(i) 
share_SH = len(df[~paid & df["city"].isin(SH)]) / len(df[df["city"].isin(SH)]) * 100
share_AL = len(df[~paid & df["city"].isin(AL)]) / len(df[df["city"].isin(AL)]) * 100
print(f"Доля отмененных заказов в Шымкенте составляет: {share_SH:.1f} %")
print(f"Доля отмененных заказов в Алмате составляет: {share_AL:.1f} %")
print(f"Шымкент выделяется в {(share_SH / share_AL):.1f} раз")

Доля отмененных заказов в Шымкенте составляет: 20.4 %
Доля отмененных заказов в Алмате составляет: 7.9 %
Шымкент выделяется в 2.6 раз


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [58]:
# 4
clean_rev_sep = df[paid & not_test & sep]
clean_rev_octo = df[paid & not_test & octo]
print(f"Выручка Шымкента за сентябрь: {(clean_rev_sep[clean_rev_sep["city"].isin(SH)]["amount"].sum()):.1f} тг, а за октябрь: {(clean_rev_octo[clean_rev_octo["city"].isin(SH)]["amount"].sum()):.1f} тг")
print(f"Выручка Алматы за сентябрь: {(clean_rev_sep[clean_rev_sep["city"].isin(AL)]["amount"].sum()):.1f} тг, а за октябрь: {(clean_rev_octo[clean_rev_octo["city"].isin(AL)]["amount"].sum()):.1f} тг")

rev_SH = (clean_rev_octo[clean_rev_octo["city"].isin(SH)]["amount"].sum() -  clean_rev_sep[clean_rev_sep["city"].isin(SH)]["amount"].sum()) / clean_rev_sep[clean_rev_sep["city"].isin(SH)]["amount"].sum() * 100
rev_AL = (clean_rev_octo[clean_rev_octo["city"].isin(AL)]["amount"].sum() -  clean_rev_sep[clean_rev_sep["city"].isin(AL)]["amount"].sum()) / clean_rev_sep[clean_rev_sep["city"].isin(AL)]["amount"].sum() * 100
print(f"Изменения доли выручки в Шымкенте: {rev_SH:.2f} %")
print(f"Изменения доли выручки в Алмате: {rev_AL:.2f} %")
print(f"Сильнее всего падение выручки у Шымкента на {(rev_SH / rev_AL):.2f} раза")

Выручка Шымкента за сентябрь: 4949800.0 тг, а за октябрь: 4044010.0 тг
Выручка Алматы за сентябрь: 15523940.0 тг, а за октябрь: 14643020.0 тг
Изменения доли выручки в Шымкенте: -18.30 %
Изменения доли выручки в Алмате: -5.67 %
Сильнее всего падение выручки у Шымкента на 3.22 раза


У нас упала выручка в Шымкенте на 3.22 раза в октябре

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [73]:
# 5
cancel_sh = df[df["city"].isin(SH) & (df["status"] == "отменён")]
print(len(cancel_sh))

164


Первая и самая критическая ошибка заключается в написаний условий, каждое условие должно быть обернуто круглыми скобками (только если это уже не готовая маска).
Вторая ошибка заключается в написаний условия содержания слова "Шымкент", в нашем DataFrame название города может быть написана в разных регистрах и на разных языках. 

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [106]:
# 6
best_client = df["customer_id"].value_counts().index[0]
print(f"Больше всего заказов сделал {best_client}")
print(df.loc[df["customer_id"] == best_client])
print("*************")
print(df.loc[df["customer_id"] == best_client]["order_id"].value_counts())

Больше всего заказов сделал C01818
      order_id        order_date customer_id       city     category   amount  \
57      101125  2026-09-01 15:00      C01818    Шымкент          Дом   7350.0   
779     100096  2026-09-09 18:00      C01818    Шымкент  Электроника  45270.0   
1371    101504  2026-09-15 21:00      C01818     Алматы       Одежда   8890.0   
1375    101504  2026-09-15 21:00      C01818     Алматы       Одежда   8890.0   
2195    101860  2026-09-24 18:00      C01818     Астана      Красота   1490.0   
2617    101462  2026-09-29 09:00      C01818     Астана  Электроника  22170.0   
2776    101495  2026-09-30 21:00      C01818     Алматы  Электроника  20180.0   
3417    103416  2026-10-08 14:00      C01818  Караганда          Дом   4330.0   
4279    102644  2026-10-18 08:00      C01818     Астана          Дом  13600.0   
4610    104001  2026-10-21 19:00      C01818     Алматы       Одежда  21990.0   
4778    104572  2026-10-23 18:00      C01818     Алматы       Одежда  1114

У нас повторяется заказ 2 раза под номером 1371 на сумму 8890 тенге, что в свою очередь немного завышает выручки в сентябре.В своё очередь, это приводит к неправильному анализу и выводу доли выручки за месяц. 

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [132]:
# 7
Q1 = df[paid & not_test]["amount"].quantile(0.25)
Q3 = df[paid & not_test]["amount"].quantile(0.75)
IQR = Q3 - Q1
boundary = Q3 + 1.5 * IQR
suspicious_orders = df[(df["amount"] > boundary) & paid & not_test]
print(f"Количество заказов выше границы {len(suspicious_orders)}")
print(f"Доля заказов выше границы {(len(suspicious_orders) / len(df[paid & not_test]) * 100):.2f} %")
print(suspicious_orders["category"].value_counts())
suspicious_orders

Количество заказов выше границы 320
Доля заказов выше границы 6.53 %
category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64


,order_id,order_date,customer_id,city,category,amount,delivery_days,status
12,102104,2026-09-01 10:00,C00273,Астана,Электроника,37970.0,1.0,оплачен
17,100762,2026-09-01 10:00,C00211,алматы,Электроника,38300.0,3.0,оплачен
21,102592,2026-09-01 11:00,C00764,Алматы,Электроника,40980.0,2.0,оплачен
49,101188,2026-09-01 15:00,C01543,Алматы,Одежда,39680.0,1.0,оплачен
51,101188,2026-09-01 15:00,C01543,Алматы,Одежда,39680.0,1.0,оплачен
...,...,...,...,...,...,...,...,...
5399,102807,2026-10-31 10:00,C01970,Актобе,Электроника,38070.0,3.0,оплачен
5430,103256,2026-10-31 15:00,C00553,Астана,Электроника,38750.0,3.0,оплачен
5436,103193,2026-10-31 16:00,C00400,Караганда,Электроника,107910.0,1.0,оплачен
5438,102846,2026-10-31 16:00,C01648,Алматы,Электроника,75340.0,3.0,оплачен


## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [157]:
# 8
print(((df[~paid]["city"].value_counts() / df["city"].value_counts()) * 100).sort_values())
print("********")
print((df[df["city"].isin(SH)]["city"].value_counts()).sort_values())

city
АЛМАТЫ        3.225806
шымкент       3.703704
 Алматы       4.545455
Almaty        5.797101
Атырау        7.534247
Алматы        7.991697
Астана        8.482871
Караганда     8.921933
Astana       10.000000
Актобе       10.327456
астана       13.095238
алматы       14.062500
Шымкент      21.518987
Shymkent     23.076923
Name: count, dtype: float64
********
city
Shymkent     39
шымкент      54
Шымкент     711
Name: count, dtype: int64


Судить о городе по одному написанию нельзя, так как города написаны в разных регистрах и на разных языках, что в свою очередь не даёт нам посмотреть на результат полноценно. Для того чтобы сделать точный анализ необходимо в столбце city привести города к их общему единому написанию, и только после этого следует делать анализ по доле отмен по городам или дальнейший анализа